# TradeFlow AI — nb3_olm_finetune (v4 — Unsloth + olmOCR-2-7B-1025)

**Fix #1**: Model ID diubah ke `allenai/olmOCR-2-7B-1025` sesuai PRD §4 Decision 2.
**Fix #3**: Menggunakan Unsloth (2× speed, 60% VRAM reduction) + LoRA rank=32 sesuai PRD §10.4.

| Phase | Data | LoRA |
|---|---|---|
| 1: Synthetic SFT | 1.500 CIPL sintetis | r=32 |
| 2: Mixed | 80% sintetis + 20% real aug | r=32 |


In [ ]:
# Unsloth dulu, baru yang lain (urutan wajib)
!pip install -q unsloth[colab-new]
!pip install -q peft datasets accelerate bitsandbytes trl qwen-vl-utils
!pip install -q pdf2image pillow python-dateutil
!apt-get update -qq && apt-get install -qq poppler-utils


In [ ]:
import json, os, gc
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from pdf2image import convert_from_path
from datasets import Dataset
from kaggle_secrets import UserSecretsClient

# FIX #1: Model ID yang benar sesuai PRD
MODEL_ID = 'allenai/olmOCR-2-7B-1025'
OUT_DIR  = Path('./olmocr-tradeflow-lora')
OUT_DIR.mkdir(parents=True, exist_ok=True)

NB0_INPUT     = Path('/kaggle/input/nb0-real-doc-augmentation')
NB1_INPUT     = Path('/kaggle/input/nb1-synthetic-generator')
MANIFEST_PATH = NB0_INPUT / 'dataset' / 'augmented_manifest.json'
SYNTHETIC_DIR = NB1_INPUT / 'dataset' / 'synthetic'
GT_PATH       = Path('/kaggle/input/tradeflow-real-docs/TradeFlow_GroundTruth_v5.2.json')

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
except:
    hf_token = None
    print('Warning: HF_TOKEN not found.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 1. Load Model dengan Unsloth (FIX #3)

In [ ]:
from unsloth import FastVisionModel

print(f'Loading {MODEL_ID} via Unsloth (2x speed, 60% VRAM reduction)...')
model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',  # Kaggle-optimised checkpointing
    token=hf_token,
)

# FIX #3: LoRA rank=32 sesuai PRD §10.4
model = FastVisionModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
)
model.print_trainable_parameters()


## 2. Dataset Preparation (Multimodal)

In [ ]:
def load_image(path_str):
    if str(path_str).endswith('.pdf'):
        pages = convert_from_path(path_str, dpi=150, first_page=1, last_page=1)
        return pages[0].convert('RGB')
    return Image.open(path_str).convert('RGB')

def create_qwen_message(img_path, json_str):
    return {
        'messages': [
            {'role':'user','content':[
                {'type':'image'},
                {'type':'text','text':'Extract CEISA fields from this shipping document. Return JSON with: nomorBl, tglBl, pelabuhan_muat, pelabuhan_bongkar, container_no, beratKotor, hs_code, namaKapal, voyageNumber.'}
            ]},
            {'role':'assistant','content':[{'type':'text','text':json_str}]}
        ],
        'image_path': img_path
    }

gt_data = json.loads(GT_PATH.read_text()) if GT_PATH.exists() else {}
dataset_examples = []

# A. Real augmented docs (TRAIN split only)
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    for item in manifest.get('train', []):
        doc_id = item['doc_id']
        if doc_id in gt_data:
            raw_path = str(item.get('path', ''))
            if raw_path.startswith('./dataset'):
                img_path = raw_path.replace('./dataset', str(NB0_INPUT / 'dataset'), 1)
            elif raw_path.startswith('/kaggle'):
                img_path = raw_path
            else:
                img_path = str(NB0_INPUT / 'dataset' / raw_path.lstrip('/'))
            fields = gt_data[doc_id].get('ceisa_fields', gt_data[doc_id])
            dataset_examples.append(create_qwen_message(img_path, json.dumps(fields)))
    print(f'Loaded {len(dataset_examples)} real augmented images.')
else:
    print(f'MANIFEST NOT FOUND: {MANIFEST_PATH}. Only synthetic data used.')

# B. Synthetic docs (300 samples for first run)
synth_count = 0
if SYNTHETIC_DIR.exists():
    for json_file in list(SYNTHETIC_DIR.glob('*.json'))[:300]:
        pdf_file = json_file.with_suffix('.pdf')
        if pdf_file.exists():
            gt_json = json.loads(json_file.read_text())
            dataset_examples.append(create_qwen_message(str(pdf_file), json.dumps(gt_json)))
            synth_count += 1
    print(f'Loaded {synth_count} synthetic documents.')

train_dataset = Dataset.from_list(dataset_examples)
print(f'Total training examples: {len(train_dataset)}')


## 3. Custom Data Collator (Qwen2-VL)

In [ ]:
class Qwen2VLDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts, images = [], []
        for ex in examples:
            texts.append(self.processor.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False))
            images.append(load_image(ex['image_path']))
        batch = self.processor(text=texts, images=images, return_tensors='pt', padding=True)
        labels = batch['input_ids'].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        batch['labels'] = labels
        return batch

data_collator = Qwen2VLDataCollator(processor)


## 4. Training (Unsloth SFTTrainer)

In [ ]:
from transformers import TrainingArguments, Trainer

# NOTE: FastVisionModel.for_training() DIHAPUS.
# Menyebabkan AttributeError: 'int' has no attribute 'mean'
# karena loop training Unsloth tidak kompatibel dengan custom Qwen2VLDataCollator.
# Unsloth tetap aktif untuk: fast loading, LoRA init, 4-bit NF4.

training_args = TrainingArguments(
    output_dir='./olmocr-tradeflow-lora/final',
    learning_rate=1e-4,
    num_train_epochs=1,             # Naikkan ke 3 untuk full training
    per_device_train_batch_size=1,  # T4 VRAM safety
    gradient_accumulation_steps=8,  # Effective batch = 8
    warmup_steps=20,               # warmup_ratio deprecated in transformers v5
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='no',
    remove_unused_columns=False,    # CRITICAL for custom collator
    fp16=True,
    optim='adamw_8bit',             # Unsloth 8-bit optimizer
    dataloader_num_workers=0,       # Disable multiprocessing untuk PDF loading
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print('=== Mulai Training Qwen2-VL via Unsloth ===')
trainer.train()
print('Training selesai. Menyimpan model...')
trainer.model.save_pretrained(OUT_DIR / 'best')


## 5. Upload ke HuggingFace Hub

In [ ]:
HF_REPO_NAME = 'muhammadghiffari/olm-ocr-cipl-v1'

if hf_token:
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=HF_REPO_NAME, exist_ok=True)
    trainer.model.push_to_hub(HF_REPO_NAME, token=hf_token)
    processor.push_to_hub(HF_REPO_NAME, token=hf_token)
    print(f'Model berhasil diupload ke: https://huggingface.co/{HF_REPO_NAME}')
else:
    print('HF_TOKEN tidak ditemukan. Simpan manual dari OUT_DIR.')
